# Fleet Pulse — Basic EDA on the raw datasets

Run this file cell-by-cell (each `# %%` is one cell, like a Jupyter notebook).
In Cursor/VS Code: click "Run Cell" above any `# %%`, or Shift+Enter inside a cell.

What it covers, for every raw table:
  1. shape            — how many rows / columns
  2. dtypes           — data type of every column
  3. head             — first rows, so you can picture the data
  4. missing values   — which columns have NaNs and how many
  5. key breakdowns   — value_counts on the important categorical columns
  6. class balance    — the failure rate the model actually has to learn

Data dictionary (what each table stands for):
  fleet_master  — 1 row per machine: the asset register (dimension table)
  telemetry     — 1 row per machine-day-sensor: daily sensor readings (long format)
  error_events  — 1 row per error-code emission
  failures      — 1 row per failure: the GROUND TRUTH the label is built from
  maintenance   — 1 row per maintenance visit (scheduled/corrective)
  tickets       — 1 row per service ticket, with free-text engineer notes

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)      # never hide columns
pd.set_option("display.width", 250)           # don't wrap wide tables
pd.set_option("display.max_colwidth", 90)     # show ticket note text

# Locate ml/data/raw by walking up from wherever we're running
# (works both as a script and inside a notebook kernel, where __file__ doesn't exist)
_start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for _dir in [_start, *_start.parents]:
    if (_dir / "data" / "raw").exists():
        DATA = _dir / "data" / "raw"
        break
else:
    raise FileNotFoundError("Couldn't find ml/data/raw — open/run this from inside the fleet-pulse repo")
print("Reading from:", DATA)

Reading from: /Users/shamanth/Desktop/Cursor/Deeksh Personal/fleet-pulse/ml/data/raw


In [2]:
fleet = pd.read_parquet(DATA / "fleet_master.parquet")
telemetry = pd.read_parquet(DATA / "telemetry")
errors = pd.read_parquet(DATA / "error_events.parquet")
failures = pd.read_parquet(DATA / "failures.parquet")
maintenance = pd.read_parquet(DATA / "maintenance.parquet")
tickets = pd.read_parquet(DATA / "tickets.parquet")

tables = {
    "fleet_master": fleet,
    "telemetry": telemetry,
    "error_events": errors,
    "failures": failures,
    "maintenance": maintenance,
    "tickets": tickets,
}
for name, df in tables.items():
    print(f"{name:<14} {df.shape[0]:>9,} rows x {df.shape[1]:>2} cols")

fleet_master         500 rows x 10 cols
telemetry      1,041,129 rows x  5 cols
error_events     323,220 rows x  5 cols
failures             440 rows x  5 cols
maintenance        1,454 rows x  4 cols
tickets            1,610 rows x  9 cols


In [3]:
for name, df in tables.items():
    print(f"\n=== {name} — {df.shape[0]:,} rows x {df.shape[1]} cols ===")
    print(df.dtypes.to_string())


=== fleet_master — 500 rows x 10 cols ===
machine_id                   str
modality                     str
model                        str
country                      str
region                       str
hospital_id                  str
hospital_name                str
install_date      datetime64[us]
scans_per_day            float64
flaky_reporter              bool

=== telemetry — 1,041,129 rows x 5 cols ===
machine_id               str
date          datetime64[us]
sensor                   str
value                float64
modality            category

=== error_events — 323,220 rows x 5 cols ===
machine_id               str
date          datetime64[us]
error_code               str
family                   str
severity                 str

=== failures — 440 rows x 5 cols ===
machine_id                  str
failure_date     datetime64[us]
component                   str
sudden                     bool
downtime_days             int64

=== maintenance — 1,454 rows x 4 cols ===
machi

In [4]:
for name, df in tables.items():
    na = df.isna().sum()
    na = na[na > 0]
    if na.empty:
        print(f"{name:<14} no missing values")
    else:
        print(f"\n{name}:")
        for col, n in na.items():
            print(f"  {col:<16} {n:>6,} missing ({n / len(df):.1%})")
# Note: NaN in maintenance.component / tickets.part_replaced is MEANINGFUL —
# preventive visits aren't tied to a component and replace no part.

fleet_master   no missing values
telemetry      no missing values
error_events   no missing values
failures       no missing values

maintenance:
  component         1,014 missing (69.7%)

tickets:
  component         1,014 missing (63.0%)
  part_replaced     1,170 missing (72.7%)


In [5]:
# 1) FLEET_MASTER — the asset register (1 row per machine)

In [6]:
fleet.head()

,machine_id,modality,model,country,region,hospital_id,hospital_name,install_date,scans_per_day,flaky_reporter
0,FP-MRI-0001,MRI,Polaris 3T,GB,EMEA,H045,City General GB,2019-11-03,34.585580,False
1,FP-MRI-0002,MRI,Meridian 3T,DE,EMEA,H040,Summit Health DE,2023-06-14,28.945707,False
2,FP-CT-0001,CT,Vela 64,US,AMER,H188,Northside US,2015-07-24,71.583220,False
3,FP-CT-0002,CT,Aquila 256,CN,APAC,H167,University Hospital CN,2019-11-19,63.336050,True
4,FP-CT-0003,CT,Aquila 256,JP,APAC,H090,Regional Medical Center JP,2024-09-10,58.522985,False


In [7]:
print("Modality mix:")
print(fleet["modality"].value_counts(), "\n")
print("By region:")
print(fleet["region"].value_counts(), "\n")
print("By country:")
print(fleet["country"].value_counts(), "\n")
print("Machine models:")
print(fleet["model"].value_counts(), "\n")
print("Flaky reporters (worse data quality):", fleet["flaky_reporter"].sum(), "machines")

Modality mix:
modality
CT      224
MRI     177
XRAY     99
Name: count, dtype: int64 

By region:
region
EMEA    196
APAC    158
AMER    146
Name: count, dtype: int64 

By country:
country
US    114
DE     72
GB     56
IN     54
FR     44
JP     42
CN     39
BR     32
ES     24
AU     23
Name: count, dtype: int64 

Machine models:
model
Vela 128         81
Aquila 256       74
Vela 64          69
Meridian 1.5T    66
Polaris 3T       62
Lumen R700       50
Meridian 3T      49
Lumen R500       49
Name: count, dtype: int64 

Flaky reporters (worse data quality): 28 machines


In [8]:
print(fleet["scans_per_day"].describe(), "\n")
print("Install dates:", fleet["install_date"].min().date(), "→", fleet["install_date"].max().date())
print("Fleet age (years):")
print(((pd.Timestamp("2025-12-31") - fleet["install_date"]).dt.days / 365.25).describe())

count    500.000000
mean      48.042012
std       20.756193
min       18.028232
25%       29.888171
50%       46.888508
75%       64.286180
max       89.851733
Name: scans_per_day, dtype: float64 

Install dates: 2015-01-05 → 2024-12-27
Fleet age (years):
count    500.000000
mean       6.328914
std        2.933113
min        1.010267
25%        3.719370
50%        6.640657
75%        8.971937
max       10.986995
Name: install_date, dtype: float64


In [9]:
# 2) TELEMETRY — daily sensor readings, LONG format (machine, day, sensor, value)

In [10]:
telemetry.head()

,machine_id,date,sensor,value,modality
0,FP-CT-0001,2025-01-01,tube_current_var,1.195832,CT
1,FP-CT-0001,2025-01-03,tube_current_var,0.918265,CT
2,FP-CT-0001,2025-01-04,tube_current_var,0.840554,CT
3,FP-CT-0001,2025-01-05,tube_current_var,1.022359,CT
4,FP-CT-0001,2025-01-06,tube_current_var,0.974153,CT


In [11]:
# Long format means each machine emits SEVERAL rows per day — one per sensor.
print("Sensors per modality:")
print(telemetry.groupby("modality", observed=True)["sensor"].unique().to_string(), "\n")
print("Rows per sensor:")
print(telemetry["sensor"].value_counts())

Sensors per modality:
modality
CT      [tube_current_var, tube_temp, detector_noise, gantry_vibration, cooling_margin, scans_...
MRI     [helium_level, compressor_temp, gradient_temp, rf_power_var, vibration_rms, chiller_fl...
XRAY                                   [filament_current, voltage_ripple, tube_temp, scans_count] 

Rows per sensor:
sensor
scans_count         174788
tube_temp           112825
tube_current_var     78044
detector_noise       78044
gantry_vibration     78044
cooling_margin       78044
helium_level         61963
compressor_temp      61963
gradient_temp        61963
rf_power_var         61963
vibration_rms        61963
chiller_flow         61963
filament_current     34781
voltage_ripple       34781
Name: count, dtype: int64


In [12]:
print("Date range:", telemetry["date"].min().date(), "→", telemetry["date"].max().date())
print("Machines reporting:", telemetry["machine_id"].nunique())

# expected days vs actual days per machine → the deliberate ~3% missing-day gap
days_per_machine = telemetry.groupby("machine_id")["date"].nunique()
total_days = (telemetry["date"].max() - telemetry["date"].min()).days + 1
print(f"\nExpected days per machine: {total_days}")
print("Actual days per machine:")
print(days_per_machine.describe())

Date range: 2025-01-01 → 2025-12-31
Machines reporting: 500

Expected days per machine: 365
Actual days per machine:
count    500.000000
mean     349.576000
std       10.560648
min      293.000000
25%      349.000000
50%      352.000000
75%      355.000000
max      361.000000
Name: date, dtype: float64


In [13]:
telemetry.groupby("sensor")["value"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
sensor,,,,,,,,
chiller_flow,61963.0,29.99,1.11,24.13,29.24,29.99,30.73,35.09
compressor_temp,61963.0,55.05,2.04,45.75,53.66,55.03,56.40,65.02
cooling_margin,78044.0,11.98,1.01,6.92,11.29,11.98,12.66,16.30
detector_noise,78044.0,1.01,0.11,0.57,0.93,1.01,1.08,1.62
filament_current,34781.0,5.00,0.14,4.40,4.91,5.00,5.10,5.63
gantry_vibration,78044.0,1.11,0.17,0.37,0.99,1.11,1.22,2.21
gradient_temp,61963.0,41.96,1.68,34.96,40.82,41.95,43.09,51.38
helium_level,61963.0,96.99,0.40,95.01,96.73,97.00,97.26,98.82
rf_power_var,61963.0,1.00,0.23,0.05,0.85,1.00,1.16,2.06


In [14]:
one = telemetry[(telemetry["machine_id"] == "FP-MRI-0001") & (telemetry["sensor"] == "helium_level")]
one = one.sort_values("date")
print(one.head(10).to_string())
print("...")
print(one.tail(5).to_string())
# Try plotting it: one.plot(x="date", y="value", title="FP-MRI-0001 helium level")

         machine_id       date        sensor      value modality
468264  FP-MRI-0001 2025-01-01  helium_level  97.390357      MRI
468265  FP-MRI-0001 2025-01-02  helium_level  97.101874      MRI
468266  FP-MRI-0001 2025-01-03  helium_level  97.446974      MRI
468267  FP-MRI-0001 2025-01-04  helium_level  97.072970      MRI
468268  FP-MRI-0001 2025-01-05  helium_level  97.403453      MRI
468269  FP-MRI-0001 2025-01-06  helium_level  97.112178      MRI
468270  FP-MRI-0001 2025-01-07  helium_level  97.798527      MRI
468271  FP-MRI-0001 2025-01-08  helium_level  97.392442      MRI
468272  FP-MRI-0001 2025-01-09  helium_level  97.488572      MRI
468273  FP-MRI-0001 2025-01-10  helium_level  96.917948      MRI
...
         machine_id       date        sensor      value modality
468610  FP-MRI-0001 2025-12-26  helium_level  97.687144      MRI
468611  FP-MRI-0001 2025-12-27  helium_level  96.881016      MRI
468612  FP-MRI-0001 2025-12-29  helium_level  96.813156      MRI
468613  FP-MRI-0001 2

In [15]:
# 3) ERROR_EVENTS — error-code emissions (mostly benign noise, by design)

In [16]:
errors.head()

,machine_id,date,error_code,family,severity
0,FP-MRI-0001,2025-01-02,SYS-102,SYS,info
1,FP-MRI-0001,2025-01-02,SYS-101,SYS,info
2,FP-MRI-0001,2025-01-03,SYS-101,SYS,info
3,FP-MRI-0001,2025-01-07,SYS-101,SYS,info
4,FP-MRI-0001,2025-01-07,SYS-101,SYS,info


In [17]:
print("Severity (note: overwhelmingly 'info' — bursts must EARN their signal):")
print(errors["severity"].value_counts(), "\n")
print("Error families:")
print(errors["family"].value_counts(), "\n")
print("Top error codes:")
print(errors["error_code"].value_counts().head(15))

Severity (note: overwhelmingly 'info' — bursts must EARN their signal):
severity
info        309589
warning      12311
critical      1320
Name: count, dtype: int64 

Error families:
family
SYS       163626
NET        91206
UI         54757
TUBE        6739
GANTRY      2000
CRYO        1925
DET         1049
GRAD         897
RF           553
PWR          468
Name: count, dtype: int64 

Top error codes:
error_code
SYS-100       54638
SYS-102       54523
SYS-101       54465
NET-102       30555
NET-100       30365
NET-101       30286
UI-100        18335
UI-101        18331
UI-102        18091
TUBE-400       2104
TUBE-401       2042
TUBE-402       1996
GANTRY-400      616
GANTRY-401      605
CRYO-400        601
Name: count, dtype: int64


In [18]:
per_machine = errors.groupby("machine_id").size()
print("Error events per machine:")
print(per_machine.describe())

Error events per machine:
count    500.000000
mean     646.440000
std       34.745834
min      573.000000
25%      623.000000
50%      644.000000
75%      667.250000
max      768.000000
dtype: float64


In [19]:
# 4) FAILURES — the ground truth (this is what we predict)

In [20]:
failures.head()

,machine_id,failure_date,component,sudden,downtime_days
0,FP-MRI-0001,2025-07-18,cold_head,False,3
1,FP-MRI-0001,2025-02-26,gradient_coil,False,4
2,FP-CT-0001,2025-04-29,xray_tube,False,4
3,FP-CT-0001,2025-03-16,detector_array,False,2
4,FP-CT-0001,2025-03-05,gantry_bearing,False,4


In [21]:
print("Failures by component:")
print(failures["component"].value_counts(), "\n")
print("Sudden failures (no precursor — unpredictable by design):")
print(failures["sudden"].value_counts(normalize=True).round(3), "\n")
print("Downtime days:")
print(failures["downtime_days"].describe())

Failures by component:
component
xray_tube         199
cold_head          63
gantry_bearing     61
detector_array     44
rf_amplifier       30
gradient_coil      28
generator          15
Name: count, dtype: int64 

Sudden failures (no precursor — unpredictable by design):
sudden
False    0.852
True     0.148
Name: proportion, dtype: float64 

Downtime days:
count    440.000000
mean       2.459091
std        1.120592
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max        4.000000
Name: downtime_days, dtype: float64


In [22]:
n_machines = fleet.shape[0]
n_weeks = 52
machine_weeks = n_machines * n_weeks
print(f"Failures in the year:        {len(failures):,}")
print(f"Machine-weeks in the year:   {machine_weeks:,}  ({n_machines} machines x {n_weeks} weeks)")
print(f"Failure rate per machine-week: {len(failures) / machine_weeks:.2%}")
print("→ ~1-2% positives = heavy class imbalance; accuracy is useless,")
print("  which is why the model reports PR-AUC / precision@k instead.")

Failures in the year:        440
Machine-weeks in the year:   26,000  (500 machines x 52 weeks)
Failure rate per machine-week: 1.69%
→ ~1-2% positives = heavy class imbalance; accuracy is useless,
  which is why the model reports PR-AUC / precision@k instead.


In [23]:
print(failures["machine_id"].value_counts().head(10))
print(f"\nMachines with ≥1 failure: {failures['machine_id'].nunique()} of {n_machines}")

machine_id
FP-CT-0033     5
FP-CT-0002     4
FP-CT-0096     4
FP-CT-0147     4
FP-CT-0001     3
FP-CT-0016     3
FP-CT-0017     3
FP-CT-0031     3
FP-MRI-0023    3
FP-MRI-0031    3
Name: count, dtype: int64

Machines with ≥1 failure: 291 of 500


In [24]:
# 5) MAINTENANCE — scheduled (preventive) vs corrective visits

In [25]:
maintenance.head()

,machine_id,date,maintenance_type,component
0,FP-MRI-0001,2025-04-29,scheduled,NaN
1,FP-MRI-0001,2025-11-07,scheduled,NaN
2,FP-MRI-0001,2025-07-21,corrective,cold_head
3,FP-MRI-0001,2025-03-02,corrective,gradient_coil
4,FP-MRI-0002,2025-02-28,scheduled,NaN


In [26]:
print(maintenance["maintenance_type"].value_counts(), "\n")
print("Corrective visits by component (should mirror failures):")
print(maintenance["component"].value_counts())

maintenance_type
scheduled     1014
corrective     440
Name: count, dtype: int64 

Corrective visits by component (should mirror failures):
component
xray_tube         199
cold_head          63
gantry_bearing     61
detector_array     44
rf_amplifier       30
gradient_coil      28
generator          15
Name: count, dtype: int64


In [27]:
# 6) TICKETS — service tickets with FREE-TEXT engineer notes (the NLP material)

In [28]:
tickets.head()

,machine_id,open_date,close_date,ticket_type,component,part_replaced,engineer_id,downtime_days,note_text
0,FP-MRI-0001,2025-07-18,2025-07-21,corrective,cold_head,helium line seal kit,ENG-GB-06,3,boil-off rate above spec (esc L2). swapped compressor adsorber (esc L2). site confirme...
1,FP-MRI-0001,2025-02-26,2025-03-02,corrective,gradient_coil,gradient amplifier board,ENG-GB-02,4,coil cooling loop pressure low. flushed coil cooling loop. customer informed.
2,FP-MRI-0001,2025-04-29,2025-04-29,preventive,NaN,NaN,ENG-GB-06,0,"routine PM done, all checks passed"
3,FP-MRI-0001,2025-11-07,2025-11-07,preventive,NaN,NaN,ENG-GB-05,0,"PM visit - filters, cal check, no issues"
4,FP-MRI-0002,2025-02-28,2025-02-28,preventive,NaN,NaN,ENG-DE-01,0,"routine pm done, all checks passed ref prev tkt"


In [29]:
print(tickets["ticket_type"].value_counts(), "\n")
print("Parts replaced (corrective only):")
print(tickets["part_replaced"].value_counts(), "\n")
print("Engineers:", tickets["engineer_id"].nunique(), "unique IDs")

ticket_type
preventive        1014
corrective         440
no_fault_found     156
Name: count, dtype: int64 

Parts replaced (corrective only):
part_replaced
oil cooling unit            63
HV cable set                56
X-ray tube                  48
slip ring brush set         33
main bearing                28
detector module             24
cold head assembly          24
helium line seal kit        21
DAS board                   20
compressor adsorber         18
RF fuse set                 17
X-ray tube insert           17
collimator assembly         15
gradient amplifier board    14
coil cooling loop           14
RF amplifier module         13
capacitor bank               8
HV generator board           7
Name: count, dtype: int64 

Engineers: 49 unique IDs


In [30]:
# ("He level low" vs "helium lvl dropping" vs "cryo issue" — motivates NLP/retrieval)
for note in tickets.loc[tickets["ticket_type"] == "corrective", "note_text"].head(12):
    print("•", note)

• boil-off rate above spec (esc L2). swapped compressor adsorber (esc L2). site confirmed ok.
• coil cooling loop pressure low. flushed coil cooling loop. customer informed.
• xray tube arc warnings. new HV cable set + oil cooling unit. customer informed.
• image noise complaint from site. replaced detector module. monitoring for 48h.
• bearing noise during rotation (esc L2). main bearing replaced. customer informed.
• spits observed during exposure ref prev tkt. tube replaced, recalibrated. system back to spec.
• sudden failure, no prior alerts. det module dropout ref prev tkt. replaced detector module ref prev tkt. customer informed.
• image noise complaint from site. das board swap, pixel map redone. QA passed, released to clinical use.
• gantry vibration high (esc L2). slip ring brushes swapped, cleaned. QA passed, released to clinical use.
• tube current unstable. tube insert swap done. site confirmed ok.
• He level low. swapped compressor adsorber. QA passed, released to clinical

In [31]:
# 7) CROSS-TABLE SANITY — one machine's full story, all tables joined by machine_id
MACHINE = "FP-MRI-0001"
print(f"=== {MACHINE} ===\n")
print(fleet[fleet["machine_id"] == MACHINE].to_string(index=False), "\n")
print("Failures:")
print(failures[failures["machine_id"] == MACHINE].to_string(index=False), "\n")
print("Maintenance:")
print(maintenance[maintenance["machine_id"] == MACHINE].to_string(index=False), "\n")
print("Tickets:")
print(tickets[tickets["machine_id"] == MACHINE][["open_date", "close_date", "ticket_type", "component", "note_text"]].to_string(index=False))
# Notice the story lines up: failure on 2025-07-18 → corrective maintenance →
# ticket with matching dates and a note describing the fix.

=== FP-MRI-0001 ===

 machine_id modality      model country region hospital_id   hospital_name install_date  scans_per_day  flaky_reporter
FP-MRI-0001      MRI Polaris 3T      GB   EMEA        H045 City General GB   2019-11-03       34.58558           False 

Failures:
 machine_id failure_date     component  sudden  downtime_days
FP-MRI-0001   2025-07-18     cold_head   False              3
FP-MRI-0001   2025-02-26 gradient_coil   False              4 

Maintenance:
 machine_id       date maintenance_type     component
FP-MRI-0001 2025-04-29        scheduled           NaN
FP-MRI-0001 2025-11-07        scheduled           NaN
FP-MRI-0001 2025-07-21       corrective     cold_head
FP-MRI-0001 2025-03-02       corrective gradient_coil 

Tickets:
 open_date close_date ticket_type     component                                                                                   note_text
2025-07-18 2025-07-21  corrective     cold_head boil-off rate above spec (esc L2). swapped compressor adsor